# 02. Data Preprocessing & Feature Engineering Pipeline
## House Price Prediction System

This notebook demonstrates the end-to-end data transformation pipeline:
1. Dropping high missing-rate features (`PoolQC`, `MiscFeature`, `Alley`, `Fence`, `FireplaceQu`).
2. Outlier removal for `GrLivArea > 4000` sq ft.
3. Log1p target transformation $\log(1 + SalePrice)$.
4. Feature engineering (`TotalSF`, `TotalBath`, `HouseAge`, `RemodAge`, `HasGarage`, `TotalPorchSF`).
5. Skewness correction for heavily skewed numeric features.
6. Ordinal and One-Hot Encoding.
7. Constructing the scikit-learn `ColumnTransformer` and verifying zero data leakage.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.data_loader import load_raw_data
from src.preprocessing import (
    remove_outliers,
    log_transform_target,
    inverse_log_transform_target,
    create_full_data_pipeline,
)
from src.config import RANDOM_SEED, TEST_SIZE, TARGET_COLUMN

### 1. Load Data & Train/Test Split (80/20)

In [ ]:
train_raw, _ = load_raw_data()
train_clean = remove_outliers(train_raw)

train_split, test_split = train_test_split(train_clean, test_size=TEST_SIZE, random_state=RANDOM_SEED)
print(f"Train split shape: {train_split.shape}")
print(f"Test split shape:  {test_split.shape}")

### 2. Fit Preprocessing Pipeline on Training Split (No Leakage)

In [ ]:
pipeline, X_train_processed_df, y_train_log = create_full_data_pipeline(train_split, scale_numeric=True)

X_test_raw = test_split.drop(columns=[TARGET_COLUMN, 'Id'], errors='ignore')
y_test_dollars = test_split[TARGET_COLUMN].values
y_test_log = log_transform_target(y_test_dollars)
X_test_processed = pipeline.transform(X_test_raw)

print(f"Processed Training Feature Matrix: {X_train_processed_df.shape}")
print(f"Processed Test Feature Matrix:     {X_test_processed.shape}")
print("Sample Engineered Features (first 5 rows):")
display(X_train_processed_df.head())